# Candidate Generation: Static Item-Item Collaborative Filtering

Personalised item-item control baseline that ranks each request by **cosine similarity** to its history articles instead of split-wide popularity. It is the comparator for the content and niche models: it uses co-click structure without content features or user embeddings, so it shows how much of a model's Recall@M can be explained by item co-occurrence alone.

Protocol:

- Build a binary user-item matrix from the training split: each user's latest `full_history_rows` plus every `candidates_train` click.
- Derive Top-`ITEM_K` cosine item-item neighbours from the matrix and cache them.
- Score each article by the summed similarity to the request's unique history articles.
- Restrict candidates to `available_from <= request timestamp` and exclude every article in the request's history.
- Break ties by training exposure popularity descending, then ascending `news_code`.
- Backfill short lists from an availability-filtered popularity order, then write the standard `impression_uid, cutoff, news_code` table and evaluate it with `cg_evaluation.py`.

The matrix is static: it is fitted once on the training split and never updated with target-period interactions. The neighbour cap `ITEM_K` defaults to 10 and the 500-candidate output is fixed in this notebook. Changing `ITEM_K` uses a separate cache file.

## 0. Configure

**This is the only cell you need to edit.** Set `REUSE_ARTIFACTS = False` to retrain the model
and rebuild every artifact. Candidate lists are always regenerated for `TARGET_SPLIT`, so a new
validation or test split is served normally.


In [ ]:
# ---- Where to run ----------------------------------------------------------------------------
# "colab" = Google Colab with Google Drive, "local" = this computer.
ENVIRONMENT = "local"  # "local" or "colab"

# ---- Experiment ------------------------------------------------------------------------------
# Split to evaluate: "test" for the final results, or "validation".
TARGET_SPLIT = "validation"

# True: load the saved model checkpoint and skip training. Serving still runs, so candidate
# lists are always regenerated for TARGET_SPLIT.
# False: retrain the model from scratch.
# Baselines without a model reuse any cached intermediate state, but their candidate lists are
# still regenerated.
REUSE_ARTIFACTS = True

# ---- Reused artifacts when REUSE_ARTIFACTS=True -----------------------------------------------
# Cached intermediate state (rebuilt only when REUSE_ARTIFACTS=False):
#   artifacts/cf/user_item_train.npz
#   artifacts/cf/matrix_manifest.json
#   artifacts/cf_item_item/item_item_cosine_topk_<ITEM_K>.npz
# Candidate lists are always regenerated.


## 1. Setup

In [ ]:
import heapq
import os
import sys
from pathlib import Path

import numpy as np
import polars as pl
from scipy import sparse


if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    CODE_ROOT = Path("/content/drive/MyDrive/submission/4.Code")
    ROOT = Path(os.environ.get("NEWSREC_ROOT", CODE_ROOT)).expanduser().resolve()
else:
    CODE_ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
    )
    ROOT = Path(os.environ.get("NEWSREC_ROOT", CODE_ROOT)).expanduser().resolve()
sys.path.insert(0, str(CODE_ROOT / "5_Models" / "5_01_Candidate_Generation"))

from cf_models import build_interaction_matrix, load_context, resolve_device
from cg_result_writer import CandidateWriter

ITEM_K = 10
M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)
REBUILD = not REUSE_ARTIFACTS
CF_DEVICE = "auto"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
RESULTS = ROOT / "results"

MODEL_NAME = "cf_item_item"
ARTIFACT_DIR = (
    ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts" / MODEL_NAME
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

## 2. Catalogue, availability and the training interaction matrix

Article rows match the shared `article_index` convention (`article_row = 1..N`); the padding row `0` is never eligible. Interactions come only from the training split: each user's latest `full_history_rows` plus every clicked article in `candidates_train`, deduplicated into a binary user x article matrix.

In [29]:
context = load_context(ROOT)
matrix, matrix_manifest = build_interaction_matrix(ROOT, context, rebuild=REBUILD)

print(
    f"Matrix: {matrix.shape[0]:,} users x {matrix.shape[1]:,} articles, "
    f"{matrix.nnz:,} nonzeros"
)
matrix_manifest

Matrix: 50,001 users x 51,283 articles, 936,005 nonzeros


{'version': 'cf-static-v1',
 'created_at': '2026-11-07T15:20:45.376471+00:00',
 'cutoff': '2019-11-12 23:59:25',
 'rows': 50001,
 'columns': 51283,
 'nonzero': 936005,
 'values': 'binary',
 'source': 'latest request_profiles_train history plus candidates_train clicks'}

## 3. Item-item cosine similarities

Each article column is scaled by `1/sqrt(count)` so a dot product between two articles equals cosine similarity. Similarities are computed blockwise as `A_norm @ A_norm.T`, the diagonal is removed, and only the Top-`ITEM_K` neighbours per article are retained; all other pairs are treated as zero.

In [30]:
def _normalised_columns(matrix):
    counts = np.asarray(matrix.sum(axis=0)).ravel().astype(np.float32)
    inverse = np.zeros_like(counts)
    nonzero = counts > 0
    inverse[nonzero] = 1.0 / np.sqrt(counts[nonzero])
    return matrix.T.tocsr().multiply(inverse[:, None]).tocsr(), counts


def _to_torch_csr(matrix, device):
    import torch

    return torch.sparse_csr_tensor(
        torch.from_numpy(matrix.indptr.astype(np.int64)).to(device),
        torch.from_numpy(matrix.indices.astype(np.int64)).to(device),
        torch.from_numpy(matrix.data.astype(np.float32)).to(device),
        size=matrix.shape,
        device=device,
        check_invariants=False,
    )


def _top_k_block(matrix, k, block):
    rows = matrix.shape[0]
    row_parts, col_parts, value_parts = [], [], []
    for start in range(0, rows, block):
        stop = min(start + block, rows)
        scores = (matrix[start:stop] @ matrix.T).tocsr()
        scores.sum_duplicates()
        scores.eliminate_zeros()
        for local in range(scores.shape[0]):
            lo, hi = scores.indptr[local], scores.indptr[local + 1]
            cols = scores.indices[lo:hi]
            values = scores.data[lo:hi]
            keep = cols != (start + local)
            cols = cols[keep]
            values = values[keep]
            if cols.size == 0:
                continue
            if cols.size > k:
                chosen = np.argpartition(values, -k)[-k:]
                cols = cols[chosen]
                values = values[chosen]
            order = np.argsort(-values, kind="stable")
            row_parts.append(np.full(cols.size, start + local, dtype=np.int32))
            col_parts.append(cols[order].astype(np.int32, copy=False))
            value_parts.append(values[order].astype(np.float32, copy=False))
    if not row_parts:
        return sparse.csr_matrix((rows, rows), dtype=np.float32)
    return sparse.csr_matrix(
        (
            np.concatenate(value_parts),
            (np.concatenate(row_parts), np.concatenate(col_parts)),
        ),
        shape=(rows, rows),
    )


def _top_k_block_gpu(matrix, k, block, device):
    import torch

    rows = matrix.shape[0]
    transpose = _to_torch_csr(matrix.T.tocsr(), device)
    row_parts, col_parts, value_parts = [], [], []
    with torch.no_grad():
        for start in range(0, rows, block):
            stop = min(start + block, rows)
            scores = torch.sparse.mm(
                _to_torch_csr(matrix[start:stop], device), transpose
            ).to_dense()
            local = torch.arange(stop - start, device=device)
            scores[local, torch.arange(start, stop, device=device)] = -1.0
            values, indices = torch.topk(scores, k, dim=1)
            values = values.cpu().numpy()
            indices = indices.cpu().numpy()
            keep = values > 0
            if not keep.any():
                continue
            row_index = np.broadcast_to(
                np.arange(start, stop, dtype=np.int32)[:, None], keep.shape
            )[keep]
            row_parts.append(row_index)
            col_parts.append(indices[keep].astype(np.int32, copy=False))
            value_parts.append(values[keep].astype(np.float32, copy=False))

    del transpose
    if not row_parts:
        return sparse.csr_matrix((rows, rows), dtype=np.float32)
    return sparse.csr_matrix(
        (
            np.concatenate(value_parts),
            (np.concatenate(row_parts), np.concatenate(col_parts)),
        ),
        shape=(rows, rows),
    )


def build_item_neighbors(matrix, k=ITEM_K, block=256, device="auto"):
    normalised, popularity = _normalised_columns(matrix)
    resolved = resolve_device(device)
    if resolved == "cuda":
        cosine = _top_k_block_gpu(normalised, k, block, resolved)
    else:
        cosine = _top_k_block(normalised, k, block)
    rows = np.repeat(np.arange(cosine.shape[0], dtype=np.int32), np.diff(cosine.indptr))
    counts = cosine.copy()
    counts.data = counts.data * np.sqrt(popularity[rows] * popularity[cosine.indices])
    return cosine, counts, popularity

## 4. Build and cache item-item neighbours

The neighbour matrix is cached as `item_item_cosine_topk_<ITEM_K>.npz` and reloaded on later runs; `REUSE_ARTIFACTS = False` (which sets `REBUILD = True`) forces recomputation. Candidate lists are always regenerated. Articles with no training interactions have no nonzero similarity row, so they can never enter the score.

In [31]:
def load_or_build_item_neighbors(matrix, k=ITEM_K, *, rebuild=False, device="auto"):
    cosine_path = ARTIFACT_DIR / f"item_item_cosine_topk_{k}.npz"
    if not rebuild and cosine_path.exists():
        return sparse.load_npz(cosine_path)

    resolved = resolve_device(device)
    cosine, _, _ = build_item_neighbors(matrix, k=k, device=resolved)
    sparse.save_npz(cosine_path, cosine, compressed=True)
    return cosine

In [32]:
item_similarity = load_or_build_item_neighbors(
    matrix, k=ITEM_K, rebuild=REBUILD, device=CF_DEVICE
)

print(f"Item-item edges: {item_similarity.nnz:,}")
print(f"Items with neighbours: {(np.diff(item_similarity.indptr) > 0).sum():,}")

Item-item edges: 355,850
Items with neighbours: 35,976


## 5. Generate the standard candidate table

Requests are processed in timestamp order because the popularity fallback advances an availability-ordered heap lazily. Each request sums the retained similarities of its unique history articles, filters by its exact `available_from <= timestamp` condition, removes history, and ranks the survivors by score descending, training exposures descending, then ascending `news_code`. Short lists are backfilled from the popularity fallback so every request receives exactly `CANDIDATE_CAP = 500` candidates.

In [33]:
class PopularityFallback:
    def __init__(self, available_from, popularity, news_code, limit=1500):
        self.popularity = popularity
        self.news_code = news_code
        self.limit = int(limit)
        order = np.argsort(available_from[1:], kind="stable") + 1
        self.order = order.astype(np.int64)
        self.order_available = available_from[self.order]
        self.position = 0
        self.heap = []

    def advance(self, timestamp):
        while (
            self.position < self.order.size
            and self.order_available[self.position] <= timestamp
        ):
            row = int(self.order[self.position])
            key = (float(self.popularity[row]), -int(self.news_code[row]), row)
            if len(self.heap) < self.limit:
                heapq.heappush(self.heap, key)
            elif key > self.heap[0]:
                heapq.heapreplace(self.heap, key)
            self.position += 1
        items = sorted(self.heap, key=lambda x: (-x[0], -x[1]))
        return np.asarray([item[2] for item in items], dtype=np.int32)


def _row_aggregate(matrix):
    totals = sparse.csr_matrix(matrix.tocsr().sum(axis=0))
    totals.sum_duplicates()
    totals.eliminate_zeros()
    return (
        totals.indices.astype(np.int32, copy=False),
        totals.data.astype(np.float32, copy=False),
    )


def select_top_k(
    candidate_rows,
    candidate_scores,
    history,
    popularity,
    news_code,
    available_from,
    timestamp,
    fallback,
    k=CANDIDATE_CAP,
):
    candidate_rows = np.asarray(candidate_rows, dtype=np.int64)
    candidate_scores = np.asarray(candidate_scores, dtype=np.float32)
    history = np.asarray(history, dtype=np.int64)
    selected = np.empty(0, dtype=np.int64)

    if candidate_rows.size:
        keep = available_from[candidate_rows] <= timestamp
        if history.size:
            keep &= ~np.isin(candidate_rows, history)
        candidate_rows = candidate_rows[keep]
        candidate_scores = candidate_scores[keep]
        if candidate_rows.size:
            order = np.lexsort(
                (
                    news_code[candidate_rows],
                    -popularity[candidate_rows],
                    -candidate_scores,
                )
            )
            selected = candidate_rows[order[:k]]

    if selected.size < k:
        remaining = np.asarray(fallback, dtype=np.int64)
        if history.size:
            remaining = remaining[~np.isin(remaining, history)]
        if selected.size:
            remaining = remaining[~np.isin(remaining, selected)]
        selected = np.concatenate([selected, remaining[: k - selected.size]])

    return selected.astype(np.int32, copy=False)


def _select_for_request(context, item_similarity, history, fallback, timestamp):
    history = np.unique(np.asarray(history, dtype=np.int64))
    if history.size:
        candidate_rows, candidate_scores = _row_aggregate(item_similarity[history])
    else:
        candidate_rows = np.empty(0, dtype=np.int32)
        candidate_scores = np.empty(0, dtype=np.float32)
    return select_top_k(
        candidate_rows,
        candidate_scores,
        history,
        context.popularity,
        context.news_code,
        context.available_from,
        timestamp,
        fallback,
    )

In [ ]:
def _load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    return pl.read_parquet(
        parts, columns=["impression_uid", "timestamp", "full_history_rows"]
    ).sort("timestamp", maintain_order=True)


def generate_recommendations(context, item_similarity, *, artifact_dir):
    path = (
        Path(artifact_dir)
        / f"recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet"
    )
    profiles = _load_profiles(TARGET_SPLIT)
    uids = profiles["impression_uid"].to_numpy()
    timestamps = profiles["timestamp"].to_numpy().astype("datetime64[ns]")
    histories = profiles["full_history_rows"].to_list()
    starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
    ends = np.r_[starts[1:], len(timestamps)]

    fallback = PopularityFallback(
        context.available_from,
        context.popularity,
        context.news_code,
        limit=CANDIDATE_CAP + 1200,
    )
    cutoffs = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
    writer = CandidateWriter(path)
    try:
        for start, end in zip(starts, ends):
            timestamp = timestamps[start]
            fallback_rows = fallback.advance(timestamp)
            selected = np.vstack(
                [
                    _select_for_request(
                        context,
                        item_similarity,
                        histories[row],
                        fallback_rows,
                        timestamp,
                    )
                    for row in range(start, end)
                ]
            )
            writer.add(
                np.repeat(uids[start:end], CANDIDATE_CAP),
                np.tile(cutoffs, end - start),
                context.news_code[selected.reshape(-1)],
            )
    finally:
        writer.close()
    return path

In [ ]:
item_path = generate_recommendations(
    context, item_similarity, artifact_dir=ARTIFACT_DIR
)
print("Item-item rows:", pl.scan_parquet(item_path).select(pl.len()).collect().item())
print("Item-item file:", item_path)

## 6. Evaluation

Uses the shared `CandidateEvaluator` and writes `results/candidate_generation_<model>_<split>.csv`. It reports recall (pooled and impression-mean), hit rate, long-tail recall, cold-article recall, novelty (bits), catalogue coverage and personalisation at each M.

In [36]:
from cg_evaluation import CandidateEvaluator

evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
results = evaluator.evaluate_to_csv(
    item_path,
    model_name=MODEL_NAME,
    m_values=M_VALUES,
    output_dir=RESULTS,
)
results.sort(["m", "model"])

model,split,m,metric,value,numerator,denominator
str,str,i64,str,f64,f64,f64
"""cf_item_item""","""validation""",50,"""catalogue_coverage""",0.596933,28377.0,47538.0
"""cf_item_item""","""validation""",50,"""cold_article_recall""",0.0,0.0,342.0
"""cf_item_item""","""validation""",50,"""hit_rate""",0.002593,82.0,31624.0
"""cf_item_item""","""validation""",50,"""long_tail_recall""",0.002864,28.0,9778.0
"""cf_item_item""","""validation""",50,"""novelty_bits""",15.705265,2.4833e7,1.5812e6
…,…,…,…,…,…,…
"""cf_item_item""","""validation""",500,"""long_tail_recall""",0.06576,643.0,9778.0
"""cf_item_item""","""validation""",500,"""novelty_bits""",15.57857,2.4633e8,1.5812e7
"""cf_item_item""","""validation""",500,"""personalisation""",0.617898,6178.976715,10000.0


## Notes and limitations

- It is personalised only through the request history: two requests with the same `full_history_rows` receive the same ranking, apart from availability and the fallback.
- The matrix is fitted once on the training split and never refreshed, so the model cannot react to drift inside validation or test.
- Similarity is truncated to the Top-`ITEM_K` neighbours per article. A candidate outside every history article's neighbour list is never scored and can only enter through the popularity fallback.
- Articles with no training interactions have no neighbours, so they are effectively unreachable; this is why `cold_article_recall` is 0.0.
- Scores are summed cosine similarities. They are not comparable across users, but ranking is within a request, so the Top-K order is unaffected.
- History articles are excluded from candidates, matching the content baselines but not the rolling popularity control, so the comparison includes a small repeat-click policy difference.
- `available_from` is the project-wide observable-availability proxy, not a verified publication timestamp.
- Use this model as the co-occurrence control when claiming that content or neural generators improve candidate recall.